# Пользовательские признаки

In [1]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("User_Features")
         .master("spark://spark-master:7077")
         .getOrCreate())

spark.conf.set("spark.sql.session.timeZone", "UTC")

### Загрузка данных

In [2]:
data_path = "/home/jovyan/work/data/task3"

ids_schema = T.StructType([
    T.StructField("user_uid", T.LongType())
])

transactions_schema = T.StructType([
    T.StructField("element_uid", T.LongType()),
    T.StructField("user_uid", T.LongType()),
    T.StructField("watched_time", T.LongType())
])

ratings_schema = T.StructType([
    T.StructField("user_uid", T.LongType()),
    T.StructField("element_uid", T.LongType()),
    T.StructField("rating", T.IntegerType()),
    T.StructField("ts", T.DoubleType())
])

df_ids_raw = (
    spark.read.option("header", "true").schema(ids_schema)
    .csv(f"{data_path}/ids.csv")
)

df_transactions = (
    spark.read.option("header", "true").schema(transactions_schema)
    .csv(f"{data_path}/cut_transactions.csv")
)

df_ratings = (
    spark.read.option("header", "true").schema(ratings_schema)
    .csv(f"{data_path}/cut_ratings.csv")
)

# Для результата оставляем по одной записи на пользователя
df_ids = (
    df_ids_raw
    .filter(F.col("user_uid").isNotNull())
    .dropDuplicates(["user_uid"])
)

df_transactions_users = df_transactions.join(
    F.broadcast(df_ids), on="user_uid", how="inner"
)

df_ratings_users = df_ratings.join(
    F.broadcast(df_ids), on="user_uid", how="inner"
)

df_ids.printSchema()
df_transactions.printSchema()
df_ratings.printSchema()

root
 |-- user_uid: long (nullable = true)

root
 |-- element_uid: long (nullable = true)
 |-- user_uid: long (nullable = true)
 |-- watched_time: long (nullable = true)

root
 |-- user_uid: long (nullable = true)
 |-- element_uid: long (nullable = true)
 |-- rating: integer (nullable = true)
 |-- ts: double (nullable = true)



### Проверка ключей и повторов

In [3]:
transaction_repeated_pairs = (
    df_transactions_users.groupBy("user_uid", "element_uid")
    .count().filter(F.col("count") > 1).count()
)

rating_repeated_pairs = (
    df_ratings_users.groupBy("user_uid", "element_uid")
    .count().filter(F.col("count") > 1).count()
)

print("Строк в выборке пользователей:", df_ids_raw.count())
print("Уникальных пользователей:", df_ids.count())
print("Повторяющихся пар user_uid + element_uid в просмотрах:", transaction_repeated_pairs)
print("Повторяющихся пар user_uid + element_uid в оценках:", rating_repeated_pairs)
print("NULL element_uid в просмотрах:", df_transactions_users.filter(F.col("element_uid").isNull()).count())
print("NULL element_uid в оценках:", df_ratings_users.filter(F.col("element_uid").isNull()).count())

Строк в выборке пользователей: 100
Уникальных пользователей: 100
Повторяющихся пар user_uid + element_uid в просмотрах: 0
Повторяющихся пар user_uid + element_uid в оценках: 0
NULL element_uid в просмотрах: 0
NULL element_uid в оценках: 0


### Последняя оценка фильма

Для каждой пары `user_uid + element_uid` оставляется оценка с максимальным `ts`. Фильтр по оценкам 8, 9 и 10 применяется после выбора последней оценки.

In [4]:
rating_window = Window.partitionBy("user_uid", "element_uid").orderBy(
    F.col("ts").desc_nulls_last()
)

# Одинаковый максимальный ts не позволяет однозначно определить последнюю оценку
df_ranked_ratings = df_ratings_users.withColumn(
    "_rank", F.rank().over(rating_window)
)

df_latest_rating_ties = (
    df_ranked_ratings.filter(F.col("_rank") == 1)
    .groupBy("user_uid", "element_uid")
    .agg(F.countDistinct("rating").alias("rating_values"))
    .filter(F.col("rating_values") > 1)
)

latest_rating_ties = df_latest_rating_ties.count()
print("Пар с несколькими записями на максимальном ts:", latest_rating_ties)

if latest_rating_ties > 0:
    df_latest_rating_ties.show(20, truncate=False)
    raise ValueError("Для некоторых фильмов последняя оценка неоднозначна: совпадает максимальный ts")

df_ratings_latest = (
    df_ratings_users
    .withColumn("_rn", F.row_number().over(rating_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

df_ratings_target = df_ratings_latest.filter(F.col("rating").isin(8, 9, 10))

df_ratings_target.groupBy("rating").count().orderBy("rating").show()

Пар с несколькими записями на максимальном ts: 0
+------+-----+
|rating|count|
+------+-----+
|     8| 6110|
|     9| 4237|
|    10| 4959|
+------+-----+



### Агрегация просмотров по фильму

Повторные просмотры одного фильма суммируются по `user_uid + element_uid` до соединения с последней оценкой.

In [5]:
df_user_film_watch = (
    df_transactions_users
    .groupBy("user_uid", "element_uid")
    .agg(F.sum("watched_time").cast("long").alias("watched_time_sum"))
)

df_user_film_rating_watch = (
    df_user_film_watch
    .join(
        df_ratings_target.select("user_uid", "element_uid", "rating"),
        on=["user_uid", "element_uid"],
        how="inner"
    )
)

### Расчёт признаков

К просмотрам присоединяется одна последняя оценка фильма; после этого признаки агрегируются по пользователю. Предварительная агрегация по фильму исключает размножение строк при JOIN.

In [6]:
feature_aggregations = []

for rating in [8, 9, 10]:
    feature_aggregations.extend([
        F.min(
            F.when(F.col("rating") == rating, F.col("watched_time_sum"))
        ).alias(f"feat_min_watched_time_rating_{rating}"),
        F.max(
            F.when(F.col("rating") == rating, F.col("watched_time_sum"))
        ).alias(f"feat_max_watched_time_rating_{rating}")
    ])

df_features_agg = (
    df_user_film_rating_watch
    .groupBy("user_uid")
    .agg(*feature_aggregations)
)

feature_columns = [
    f"feat_{agg}_watched_time_rating_{rating}"
    for rating in [8, 9, 10]
    for agg in ["min", "max"]
]

df_user_features = (
    df_ids
    .select("user_uid")
    .join(df_features_agg, on="user_uid", how="left")
    .select("user_uid", *feature_columns)
)

df_user_features.printSchema()
df_user_features.show(20, truncate=False)

root
 |-- user_uid: long (nullable = true)
 |-- feat_min_watched_time_rating_8: long (nullable = true)
 |-- feat_max_watched_time_rating_8: long (nullable = true)
 |-- feat_min_watched_time_rating_9: long (nullable = true)
 |-- feat_max_watched_time_rating_9: long (nullable = true)
 |-- feat_min_watched_time_rating_10: long (nullable = true)
 |-- feat_max_watched_time_rating_10: long (nullable = true)

+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+
|user_uid|feat_min_watched_time_rating_8|feat_max_watched_time_rating_8|feat_min_watched_time_rating_9|feat_max_watched_time_rating_9|feat_min_watched_time_rating_10|feat_max_watched_time_rating_10|
+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+------------------------------

### Проверка результата

In [7]:
user_movie_stats = (
    df_user_film_rating_watch
    .groupBy("user_uid")
    .agg(
        F.count("*").alias("rated_movies"),
        F.countDistinct("rating").alias("rating_categories")
    )
)

df_user_examples = (
    df_ids
    .select("user_uid")
    .join(user_movie_stats, on="user_uid", how="left")
    .fillna({"rated_movies": 0, "rating_categories": 0})
    .join(df_user_features, on="user_uid", how="inner")
)

print("Пользователей в исходной выборке:", df_ids.count())
print("Строк в результате:", df_user_features.count())
print("Уникальных пользователей в результате:", df_user_features.select("user_uid").distinct().count())

# Для пользователей без подходящих просмотров все признаки должны быть NULL
all_features_null = None
for column in feature_columns:
    condition = F.col(column).isNull()
    all_features_null = condition if all_features_null is None else (all_features_null & condition)

no_rated_movies_with_features = df_user_examples.filter(
    (F.col("rated_movies") == 0) & (~all_features_null)
).count()

invalid_min_max = df_user_features.filter(
    (F.col("feat_min_watched_time_rating_8") > F.col("feat_max_watched_time_rating_8")) |
    (F.col("feat_min_watched_time_rating_9") > F.col("feat_max_watched_time_rating_9")) |
    (F.col("feat_min_watched_time_rating_10") > F.col("feat_max_watched_time_rating_10"))
).count()

print("Пользователей без подходящих фильмов:", df_user_examples.filter(F.col("rated_movies") == 0).count())
print("Нарушений NULL для пользователей без подходящих фильмов:", no_rated_movies_with_features)
print("Нарушений min <= max:", invalid_min_max)

assert df_user_features.count() == df_ids.count()
assert df_user_features.select("user_uid").distinct().count() == df_user_features.count()
assert no_rated_movies_with_features == 0
assert invalid_min_max == 0

display(
    df_user_examples.groupBy("rated_movies", "rating_categories")
    .count()
    .orderBy("rated_movies", "rating_categories")
)
for label, condition in [
    ("Без подходящих фильмов", F.col("rated_movies") == 0),
    ("Один фильм с целевой оценкой", F.col("rated_movies") == 1),
    ("Несколько фильмов с целевыми оценками", F.col("rated_movies") > 1)
]:
    print(label)
    display(df_user_examples.filter(condition).limit(3))

Пользователей в исходной выборке: 100
Строк в результате: 100
Уникальных пользователей в результате: 100
Пользователей без подходящих фильмов: 41
Нарушений NULL для пользователей без подходящих фильмов: 0
Нарушений min <= max: 0


DataFrame[rated_movies: bigint, rating_categories: bigint, count: bigint]

Без подходящих фильмов


DataFrame[user_uid: bigint, rated_movies: bigint, rating_categories: bigint, feat_min_watched_time_rating_8: bigint, feat_max_watched_time_rating_8: bigint, feat_min_watched_time_rating_9: bigint, feat_max_watched_time_rating_9: bigint, feat_min_watched_time_rating_10: bigint, feat_max_watched_time_rating_10: bigint]

Один фильм с целевой оценкой


DataFrame[user_uid: bigint, rated_movies: bigint, rating_categories: bigint, feat_min_watched_time_rating_8: bigint, feat_max_watched_time_rating_8: bigint, feat_min_watched_time_rating_9: bigint, feat_max_watched_time_rating_9: bigint, feat_min_watched_time_rating_10: bigint, feat_max_watched_time_rating_10: bigint]

Несколько фильмов с целевыми оценками


DataFrame[user_uid: bigint, rated_movies: bigint, rating_categories: bigint, feat_min_watched_time_rating_8: bigint, feat_max_watched_time_rating_8: bigint, feat_min_watched_time_rating_9: bigint, feat_max_watched_time_rating_9: bigint, feat_min_watched_time_rating_10: bigint, feat_max_watched_time_rating_10: bigint]

### Сохранение результата

In [8]:
output_path = "s3a://dwh/task3/user_features/"

df_user_features.write.mode("overwrite").parquet(output_path)

df_saved = spark.read.parquet(output_path)
print("Путь сохранения:", output_path)
print("Сохранено строк:", df_saved.count())
df_saved.printSchema()

Путь сохранения: s3a://dwh/task3/user_features/
Сохранено строк: 100
root
 |-- user_uid: long (nullable = true)
 |-- feat_min_watched_time_rating_8: long (nullable = true)
 |-- feat_max_watched_time_rating_8: long (nullable = true)
 |-- feat_min_watched_time_rating_9: long (nullable = true)
 |-- feat_max_watched_time_rating_9: long (nullable = true)
 |-- feat_min_watched_time_rating_10: long (nullable = true)
 |-- feat_max_watched_time_rating_10: long (nullable = true)

